In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from AdaptiveBILSTM import BiLSTMPredictor  # 复用原有的预测器

class TimeSeriesEncoder_NoFeatureExtraction(nn.Module):
    """
    消融版编码器：
    - 移除了 MultiScaleEncoder
    - 移除了 TemporalDependency
    - 移除了 FeatureFusion
    - 保留了 Gaussian Domain Adaptation (贝叶斯域适应)
    """
    def __init__(self, input_dim, hidden_dim, num_domains=7, num_layers=2, dropout=0.3):
        super().__init__()
        self.input_dim = input_dim
        # 确保 hidden_dim 能被 4 整除 (保持与原模型一致的维度逻辑)
        self.hidden_dim = (hidden_dim // 4) * 4
        self.num_domains = num_domains
        
        # 1. 简单的特征投影 (替代原本复杂的特征提取)
        # 将输入 [input_dim] 映射到 [hidden_dim]
        self.simple_feature_projection = nn.Sequential(
            nn.Linear(input_dim, self.hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.LayerNorm(self.hidden_dim)
        )
        
        # 2. 贝叶斯域自适应参数 (保持不变)
        self.domain_mu = nn.Parameter(torch.zeros(num_domains, self.hidden_dim))
        self.domain_logvar = nn.Parameter(torch.zeros(num_domains, self.hidden_dim))
        self.domain_importance = nn.Parameter(torch.ones(num_domains))
        
        # 3. 域适应基础网络 (保持不变)
        self.domain_adapter_base = nn.Sequential(
            nn.Linear(self.hidden_dim, self.hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
    
    def gaussian_domain_adapt(self, features, domain_idx=None):
        """保持原有的贝叶斯域适应逻辑不变"""
        if domain_idx is not None:
            mu = self.domain_mu[domain_idx]
            logvar = self.domain_logvar[domain_idx]
            importance = F.softplus(self.domain_importance[domain_idx])
            
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            domain_params = mu + eps * std
            
            adapted_features = features * (domain_params * importance.unsqueeze(-1))
        else:
            domain_weights = F.softmax(self.domain_importance, dim=0)
            mixed_mu = torch.sum(self.domain_mu * domain_weights.unsqueeze(1), dim=0)
            mixed_logvar = torch.sum(self.domain_logvar * domain_weights.unsqueeze(1), dim=0)
            
            std = torch.exp(0.5 * mixed_logvar)
            eps = torch.randn_like(std)
            domain_params = mixed_mu + eps * std
            
            adapted_features = features * domain_params
        
        return self.domain_adapter_base(adapted_features)
    
    def forward(self, x, domain_idx=None):
        batch_size, num_buildings, seq_len, _ = x.shape
        
        # 调整形状: [B*N, T, D]
        x_reshaped = x.view(batch_size * num_buildings, seq_len, self.input_dim)
        
        # --- 变化点：不再调用多尺度和注意力，直接投影 ---
        base_features = self.simple_feature_projection(x_reshaped)
        
        # 贝叶斯域适应 (保持不变)
        adapted_features = []
        for t in range(seq_len):
            t_feat = base_features[:, t, :]
            t_adapted = self.gaussian_domain_adapt(t_feat, domain_idx)
            adapted_features.append(t_adapted)
        adapted_features = torch.stack(adapted_features, dim=1)
    
        return adapted_features.view(batch_size, num_buildings, seq_len, self.hidden_dim)


class AdaptiveBiLSTM_NoFeatureExtraction(nn.Module):
    """消融实验主模型：无复杂特征提取"""
    def __init__(self, input_dim, hidden_dim, category_dim, forecast_horizon, 
                 num_buildings, num_domains=7, num_layers=2, dropout=0.3):
        super().__init__()
        
        # 使用简化的 Encoder
        self.time_series_encoder = TimeSeriesEncoder_NoFeatureExtraction(
            input_dim=input_dim,
            hidden_dim=hidden_dim,
            num_domains=num_domains,
            num_layers=num_layers,
            dropout=dropout
        )
        
        # 预测器完全复用原模型
        self.bilstm_predictor = BiLSTMPredictor(
            input_dim=hidden_dim, # 注意这里 encoder 输出已经是 hidden_dim
            hidden_dim=hidden_dim,
            category_dim=category_dim,
            forecast_horizon=forecast_horizon,
            num_buildings=num_buildings,
            num_layers=num_layers,
            dropout=dropout
        )
    
    def forward(self, x, category, domain_idx=None):
        # 编码 (含贝叶斯域适应，但不含复杂特征提取)
        time_features = self.time_series_encoder(x, domain_idx)
        
        # 预测
        predictions = self.bilstm_predictor(time_features, category)
        
        return predictions

In [2]:
# 定义Huber损失函数
class HuberLoss(nn.Module):
    def __init__(self, delta=1.0):
        super(HuberLoss, self).__init__()
        self.delta = delta
    
    def forward(self, predictions, targets):
        # 计算预测值与目标值之间的差异
        diff = predictions - targets
        abs_diff = torch.abs(diff)
        
        # 应用Huber损失公式
        loss = torch.where(
            abs_diff <= self.delta,
            0.5 * diff * diff,
            self.delta * (abs_diff - 0.5 * self.delta)
        )
        
        return torch.mean(loss)

# 计算损失的辅助函数
def calculate_loss(predictions, targets):
    """
    计算Huber损失
    
    参数:
    - predictions: 预测值
    - targets: 真实值
    
    返回:
    - loss: Huber损失值
    """
    loss_fn = HuberLoss(delta=1.0)
    return loss_fn(predictions, targets)

def train_and_save_model(model, train_loader, val_loader, epochs, lr, weight_decay, 
                       model_name, save_dir='models', device='cuda', early_stopping_patience=5,
                       source_domain_idx=0, target_domain_idx=None):
    """
    训练模型并保存最佳模型，支持域自适应
    
    新增参数:
    - source_domain_idx: 源域索引，默认为0
    - target_domain_idx: 用于验证，默认为0，使用源域验证
    """
    os.makedirs(save_dir, exist_ok=True)
    model = model.to(device)
    
    # 检查train_loader是否为None（CC类别的情况）
    if train_loader is None:
        print(f"⚠️ 警告：训练数据加载器为空（可能是CC类别），跳过训练")
        return model, None, None
    
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
    
    # 初始化记录器
    train_losses = []
    val_losses = []
    best_val_loss = float('inf')
    best_epoch = 0
    patience_counter = 0
    
    # 保存模型配置信息（不同模型有不同的属性）
    model_config = {}
    
    # 尝试获取常见模型属性（如果存在的话）
    for attr in ['input_dim', 'hidden_dim', 'forecast_horizon', 'num_buildings', 'num_layers', 'dropout']:
        if hasattr(model, attr):
            model_config[attr] = getattr(model, attr)
    
    # 根据模型类型获取特定属性
    is_adaptive_model = hasattr(model, 'time_series_encoder')
    if is_adaptive_model:
        model_config['model_type'] = 'AdaptiveBiLSTM'
        if hasattr(model.time_series_encoder, 'num_domains'):
            model_config['num_domains'] = model.time_series_encoder.num_domains
        model_config['source_domain_idx'] = source_domain_idx
        model_config['target_domain_idx'] = target_domain_idx
    
    # 保存最佳模型的信息
    best_model_info = {
        'state_dict': None,
        'optimizer_state': None,
        'epoch': 0,
        'train_loss': float('inf'),
        'val_loss': float('inf'),
        'metrics': None
    }

    print(f"开始训练 {model_name}...")

    for epoch in range(epochs):
        # 训练阶段
        model.train()
        epoch_train_loss = 0
        train_steps = 0
        
        progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs} [Training]', mininterval=3.0)
        for batch in progress_bar:
            # 修改：解包新的数据格式 (features, targets, category_name, category_onehot)
            inputs, targets, category_name, category_onehot = batch
            
            # 将数据移动到设备
            inputs = inputs.float().to(device)
            targets = targets.float().to(device)
            category_onehot = category_onehot.float().to(device)
            
            optimizer.zero_grad()
            
            # 根据模型类型调用
            if is_adaptive_model:
                predictions = model(inputs, category_onehot, domain_idx=source_domain_idx)
            else:
                predictions = model(inputs, category_onehot)
                
            loss = calculate_loss(predictions, targets)  # 使用Huber损失函数
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            epoch_train_loss += loss.item()
            train_steps += 1
            progress_bar.set_postfix({'train_loss': f'{loss.item():.4f}'})
        
        avg_train_loss = epoch_train_loss / train_steps
        train_losses.append(avg_train_loss)
        
        # 验证阶段
        if val_loader is not None:
            model.eval()
            epoch_val_loss = 0
            val_steps = 0
            
            with torch.no_grad():
                progress_bar = tqdm(val_loader, desc=f'Epoch {epoch+1}/{epochs} [Validation]')
                for batch in progress_bar:
                    # 修改：解包新的数据格式
                    inputs, targets, category_name, category_onehot = batch
                    
                    inputs = inputs.float().to(device)
                    targets = targets.float().to(device)
                    category_onehot = category_onehot.float().to(device)
                    
                    if is_adaptive_model:
                        predictions = model(inputs, category_onehot, domain_idx=source_domain_idx)
                    else:
                        predictions = model(inputs, category_onehot)
                    
                    loss = calculate_loss(predictions, targets)
                    
                    epoch_val_loss += loss.item()
                    val_steps += 1
                    progress_bar.set_postfix({'val_loss': f'{loss.item():.4f}'})
            
            avg_val_loss = epoch_val_loss / val_steps
            val_losses.append(avg_val_loss)
            
            # 学习率调整
            scheduler.step(avg_val_loss)
            
            # 计算当前模型的评估指标
            current_metrics = simple_evaluate_model(
                model, val_loader, f"{model_name}_epoch_{epoch+1}", 
                device=device, domain_idx=target_domain_idx if is_adaptive_model else None
            )
            
            # 检查是否是最佳模型
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                best_epoch = epoch
                patience_counter = 0
                
                # 更新最佳模型信息
                best_model_info = {
                    'state_dict': copy.deepcopy(model.state_dict()),
                    'optimizer_state': copy.deepcopy(optimizer.state_dict()),
                    'epoch': epoch + 1,
                    'train_loss': avg_train_loss,
                    'val_loss': avg_val_loss,
                    'metrics': current_metrics,
                    'hyperparameters': {
                        'lr': lr,
                        'weight_decay': weight_decay,
                        'epochs': epochs,
                        'best_epoch': epoch + 1,
                        'model_config': model_config,
                        'model_type': type(model).__name__,
                        'source_domain_idx': source_domain_idx if is_adaptive_model else None,
                        'target_domain_idx': target_domain_idx if is_adaptive_model else None
                    }
                }
                
                # 保存最佳模型检查点
                checkpoint_path = os.path.join(save_dir, f'{model_name}_best.pth')
                torch.save(best_model_info, checkpoint_path)
                print(f"✅ 保存最佳模型 (epoch {epoch+1}), 验证损失: {avg_val_loss:.4f}")
            else:
                patience_counter += 1
            
            # 打印当前epoch的训练信息
            print(
                f"Epoch {epoch+1}/{epochs} - "
                f"Train Loss: {avg_train_loss:.4f}, "
                f"Val Loss: {avg_val_loss:.4f}, "
                f"RMSD: {current_metrics['RMSD']:.4f}, "
                f"R²: {current_metrics['R2']:.4f}, "
                f"Best Val Loss: {best_val_loss:.4f} (Epoch {best_epoch+1}), "
                f"LR: {optimizer.param_groups[0]['lr']:.6f}"
            )
            
            # 早停检查
            if patience_counter >= early_stopping_patience:
                print(f"Early stopping triggered after epoch {epoch+1}")
                break
        else:
            # 没有验证集时只打印训练损失
            print(f"Epoch {epoch+1}/{epochs} - Train Loss: {avg_train_loss:.4f}")
    
    # 训练结束后，保存训练历史
    history = {
        'train_losses': train_losses,
        'val_losses': val_losses,
        'best_epoch': best_epoch + 1,
        'best_val_loss': best_val_loss
    }
    
    # 保存训练历史
    history_path = os.path.join(save_dir, f'{model_name}_training_history.json')
    with open(history_path, 'w') as f:
        serializable_history = {
            'train_losses': [float(loss) for loss in train_losses],
            'val_losses': [float(loss) for loss in val_losses],
            'best_epoch': best_epoch + 1,
            'best_val_loss': float(best_val_loss)
        }
        json.dump(serializable_history, f, indent=4)
    
    # 绘制训练曲线（如果有验证集）
    if val_losses:
        plt.figure(figsize=(10, 5))
        plt.plot(train_losses, label='Train Loss')
        plt.plot(val_losses, label='Validation Loss')
        plt.axvline(x=best_epoch, color='r', linestyle='--', label=f'Best Model (Epoch {best_epoch+1})')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title(f'{model_name} Training History')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.savefig(os.path.join(save_dir, f'{model_name}_training_curve.png'))
        plt.close()
    
    # 恢复最佳模型状态
    if best_model_info['state_dict'] is not None:
        model.load_state_dict(best_model_info['state_dict'])
    
    return model, best_model_info, history
# 修改简化评估函数
def simple_evaluate_model(model, test_loader, model_name="Model", device='cuda', domain_idx=None):
    """
    训练中使用的简化评估函数，支持域自适应
    """
    from sklearn.metrics import mean_squared_error, r2_score
    
    if test_loader is None:
        print("⚠️ 测试数据加载器为空")
        return {
            "Model": model_name,
            "RMSD": float('inf'),
            "R2": 0.0,
            "MAPE": float('inf'),
            "CC": 0.0
        }
    
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for batch in test_loader:
            # 修改：解包新的数据格式
            inputs, targets, category_name, category_onehot = batch
            
            inputs = inputs.float().to(device)
            targets = targets.float().to(device)
            category_onehot = category_onehot.float().to(device)
            
            # 根据是否是自适应模型选择调用方式
            if hasattr(model, 'time_series_encoder'):
                predictions = model(inputs, category_onehot, domain_idx=domain_idx)
            else:
                predictions = model(inputs, category_onehot)
            
            # 收集预测和目标值
            pred_values = predictions.cpu().numpy()
            target_values = targets.cpu().numpy()
            
            all_preds.append(pred_values)
            all_targets.append(target_values)
    
    # 合并所有批次的预测和目标
    all_preds = np.concatenate(all_preds, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)
    
    # 展平数组以便计算指标
    all_preds = all_preds.reshape(-1)
    all_targets = all_targets.reshape(-1)
    
    # 计算基本指标
    mse = mean_squared_error(all_targets, all_preds)
    rmsd = np.sqrt(mse)
    r2 = r2_score(all_targets, all_preds)
    
    # 计算MAPE（排除零值）
    mask = np.abs(all_targets) > 1e-6
    mape = np.mean(np.abs((all_targets[mask] - all_preds[mask]) / all_targets[mask])) * 100 if np.any(mask) else np.nan
    
    # 计算CC (相关系数)
    cc = np.corrcoef(all_preds, all_targets)[0, 1]
    
    return {
        "Model": model_name,
        "RMSD": rmsd,
        "R2": r2,
        "MAPE": mape,
        "CC": cc
    }


In [3]:
import traceback
import logging
import numpy as np
from tqdm import tqdm
import warnings
import torch.nn.functional as F
import copy
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.autograd import Function
from datetime import datetime

# 梯度反转层 - 域对抗训练的核心组件
class GradientReversalFunction(Function):
    """
    梯度反转层 - 在反向传播时反转梯度方向，实现域对抗训练
    前向传播：直接传递输入
    反向传播：将梯度乘以负的alpha值，实现梯度反转
    """
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha  # 存储alpha参数用于反向传播
        return x.view_as(x)  # 前向传播不改变输入
    
    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.alpha, None  # 反向传播时反转梯度方向

def grad_reverse(x, alpha=1.0):
    """梯度反转函数的包装，便于调用"""
    return GradientReversalFunction.apply(x, alpha)

# 域判别器 - 用于区分源域和目标域特征
class DomainDiscriminator(nn.Module):
    """
    域判别器 - 用于区分源域和目标域特征
    
    Args:
        feature_dim: 输入特征维度
        hidden_dim: 隐藏层维度
        dropout: Dropout比率
    """
    def __init__(self, feature_dim, hidden_dim=64, dropout=0.3):
        super(DomainDiscriminator, self).__init__()
        self.feature_dim = feature_dim
        
        # 简单判别器模型
        self.simple_model = nn.Sequential(
            nn.Linear(feature_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )
    
    def forward(self, x, alpha=1.0):
        # 应用梯度反转
        reversed_x = grad_reverse(x, alpha)
        return self.simple_model(reversed_x)

# Huber损失 - 对异常值更鲁棒的损失函数
class HuberLoss(nn.Module):
    """
    Huber损失 - 对异常值更鲁棒的损失函数
    
    Args:
        delta: 阈值参数，控制MSE和MAE之间的平滑过渡
    """
    def __init__(self, delta=1.0):
        super(HuberLoss, self).__init__()
        self.delta = delta
    
    def forward(self, predictions, targets):
        # 计算预测值与目标值之间的差异
        diff = predictions - targets
        abs_diff = torch.abs(diff)
        
        # 应用Huber损失公式
        loss = torch.where(
            abs_diff <= self.delta,
            0.5 * diff * diff,
            self.delta * (abs_diff - 0.5 * self.delta)
        )
        
        return torch.mean(loss)

def domain_adversarial_training_step(
    source_features, 
    target_features, 
    domain_discriminator, 
    optimizer_disc, 
    epoch, 
    total_epochs,
    device,
    projection_layer=None,
    target_domain_idx=1  # 目标域索引（仅用于数据标识，不影响二分类标签）
):
    """
    执行单个域对抗训练步骤（二分类逻辑，源域vs目标域）
    
    Args:
        source_features: 源域特征 [batch, buildings, seq_len, features] 或 [batch, seq_len, features]
        target_features: 目标域特征 [batch, buildings, seq_len, features] 或 [batch, seq_len, features]
        domain_discriminator: 域判别器模型
        optimizer_disc: 域判别器优化器
        epoch: 当前训练轮次
        total_epochs: 总训练轮次
        device: 计算设备
        projection_layer: 特征投影层（用于维度匹配）
        target_domain_idx: 目标域的索引值（仅用于数据层面的标识，标签统一为1）
    """
    try:
        # 打印输入特征形状用于调试
        logging.debug(f"Source features shape: {source_features.shape}")
        logging.debug(f"Target features shape: {target_features.shape}")
        
        # 处理4D特征格式 [batch, buildings, seq_len, features]
        if len(source_features.shape) == 4:
            batch_size, num_buildings, seq_len, feature_dim = source_features.shape
            # 将buildings维度合并到batch中
            source_features = source_features.reshape(batch_size * num_buildings, seq_len, feature_dim)
            target_features = target_features.reshape(batch_size * num_buildings, seq_len, feature_dim)
            batch_size = source_features.size(0)  # 更新批次大小
        else:
            batch_size = source_features.size(0)
        
        # 确保特征维度匹配
        if source_features.size(-1) != target_features.size(-1):
            raise ValueError(f"特征维度不匹配: 源域={source_features.size(-1)}, 目标域={target_features.size(-1)}")
        
        # 分离特征以避免重复反向传播
        source_features = source_features.detach()
        target_features = target_features.detach()
        
        # 准备二分类标签（源域=0，目标域=1）
        source_domain_labels = torch.zeros(batch_size, 1).to(device)
        target_domain_labels = torch.ones(batch_size, 1).to(device)
        
        # 特征降维处理 - 在时间维度上取平均
        if len(source_features.shape) == 3:  # [batch, seq_len, hidden_dim]
            source_processed = source_features.mean(dim=1)  # [batch, hidden_dim]
            target_processed = target_features.mean(dim=1)  # [batch, hidden_dim]
        else:
            # 如果还有其他维度，先展平再处理
            source_processed = source_features.reshape(batch_size, -1)
            target_processed = target_features.reshape(batch_size, -1)
        
        # 维度匹配处理
        expected_dim = domain_discriminator.feature_dim
        if source_processed.size(-1) != expected_dim:
            if projection_layer is not None:
                source_processed = projection_layer(source_processed)
                target_processed = projection_layer(target_processed)
            else:
                # 动态创建投影层（如果未提供）
                if not hasattr(domain_adversarial_training_step, 'projection_layer'):
                    domain_adversarial_training_step.projection_layer = nn.Linear(
                        source_processed.size(-1), expected_dim
                    ).to(device)
                source_processed = domain_adversarial_training_step.projection_layer(source_processed)
                target_processed = domain_adversarial_training_step.projection_layer(target_processed)
        
        # 连接特征和标签
        features = torch.cat([source_processed, target_processed], dim=0)
        domain_labels = torch.cat([source_domain_labels, target_domain_labels], dim=0)
        
        # 梯度反转参数 - 随训练进度增加
        grad_reverse_strength = 2. / (1. + np.exp(-10 * epoch / total_epochs)) - 1
        reversed_features = grad_reverse(features, grad_reverse_strength)
        
        # 域判别器预测
        domain_preds = domain_discriminator.simple_model(reversed_features)
        
        # 计算二分类损失
        domain_loss = F.binary_cross_entropy_with_logits(domain_preds, domain_labels)
        
        # 更新判别器
        optimizer_disc.zero_grad()
        domain_loss.backward(retain_graph=True)
        optimizer_disc.step()
        
        # 分离预测结果
        source_domain_preds = domain_preds[:batch_size]
        target_domain_preds = domain_preds[batch_size:]
        
        return domain_loss, source_domain_preds, target_domain_preds
        
    except Exception as e:
        logging.warning(f"域对抗训练步骤出错: {str(e)}")
        traceback.print_exc()
        return (
            torch.tensor(0.0, device=device),
            torch.zeros(batch_size, 1, device=device),
            torch.zeros(batch_size, 1, device=device)
        )

# 自适应λ调度器 - 动态调整域对抗训练强度
def adaptive_lambda_scheduler(epoch, epochs, source_loss, target_loss, domain_loss, lambda_domain):
    """
    基于训练进度、任务损失和域判别损失动态调整梯度反转参数λ
    
    Args:
        epoch: 当前训练轮次
        epochs: 总训练轮次
        source_loss: 源域任务损失
        target_loss: 目标域任务损失
        domain_loss: 域判别损失
        lambda_domain: 基础λ值
    
    Returns:
        float: 调整后的λ值
    """
    # 1. 基于训练进度的基础调整
    progress = epoch / epochs
    
    # 训练初期：较小的λ值，专注于任务学习
    if progress < 0.3:
        base_lambda = max(0.001, lambda_domain * 0.01)
    # 训练中期：中等λ值，平衡任务学习和域适应
    elif progress < 0.7:
        base_lambda = max(0.005, lambda_domain * 0.05)
    # 训练后期：较大λ值，加强域适应
    else:
        base_lambda = max(0.01, lambda_domain * 0.1)
    
    # 2. 基于源域和目标域任务损失比例的调整
    task_ratio = target_loss / (source_loss + 1e-10)
    
    # 如果目标域损失远大于源域，减小λ以专注于任务学习
    if task_ratio > 2.0:
        adjust_factor = 0.5
    # 如果目标域损失远小于源域，增大λ以加强域适应
    elif task_ratio < 0.5:
        adjust_factor = 2.0
    else:
        adjust_factor = 1.0
    
    # 3. 基于域判别器性能的调整
    # 如果判别器损失接近0.693(log(2))，说明判别器无法区分域，减小λ
    if abs(domain_loss - 0.693) < 0.1:
        domain_factor = 0.8
    # 如果判别器损失太小，说明判别器过强，增大λ
    elif domain_loss < 0.3:
        domain_factor = 1.5
    else:
        domain_factor = 1.0
    
    # 计算最终λ值并限制在合理范围内
    final_lambda = base_lambda * adjust_factor * domain_factor
    return min(max(final_lambda, 0.001), 0.1)  # 限制范围 [0.001, 0.1]

import traceback as tb  # ✅ 在文件顶部导入，使用别名避免冲突

def adapt_to_target_domain(source_model, source_loader, target_loader, epochs=10, lr=0.001, 
                          device='cuda', lambda_domain=0.4, early_stopping_patience=5,
                          source_domain_idx=0, target_domain_idx=1):
    """
    修复版：域自适应迁移学习，处理不同形状的数据
    """
    
    if source_loader is None or target_loader is None:
        logging.error("数据加载器为空")
        return None, None
    
    os.makedirs('models', exist_ok=True)
    model = copy.deepcopy(source_model).to(device)
    is_adaptive_model = hasattr(model, 'time_series_encoder')
    
    # ✅ 修复：安全获取特征维度
    feature_dim = 64  # 默认值
    try:
        # 尝试从目标域获取样本（通常更简单）
        for batch in target_loader:
            sample_inputs = batch[0].float().to(device)
            sample_category = batch[3].float().to(device) if len(batch) > 3 else None
            
            with torch.no_grad():
                if is_adaptive_model:
                    sample_features = model.time_series_encoder(sample_inputs, domain_idx=target_domain_idx)
                else:
                    sample_features = model(sample_inputs, sample_category)
            
            feature_dim = sample_features.shape[-1]
            logging.info(f"从目标域确定特征维度: {feature_dim}")
            break
    except Exception as e:
        logging.warning(f"特征维度确定失败，使用默认值 {feature_dim}: {e}")
    
    # 初始化组件
    domain_discriminator = DomainDiscriminator(feature_dim=feature_dim, hidden_dim=64, dropout=0.3).to(device)
    projection_layer = nn.Linear(feature_dim, feature_dim).to(device)
    
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    optimizer_disc = optim.AdamW(domain_discriminator.parameters(), lr=lr*0.5, weight_decay=0.02)
    optimizer_proj = optim.Adam(projection_layer.parameters(), lr=lr)
    
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2, eta_min=lr*0.1)
    
    huber_loss_fn = HuberLoss(delta=1.0)
    
    best_rmse = float('inf')
    patience_counter = 0
    best_model_state = None
    history = {'total_loss': [], 'task_loss': [], 'domain_loss': [], 'rmse': []}
    
    epoch_source_losses, epoch_target_losses, epoch_domain_losses = [], [], []
    
    logging.info("开始域自适应迁移学习...")
    
    for epoch in range(epochs):
        model.train()
        domain_discriminator.train()
        
        # 动态λ
        if epoch > 0 and epoch_source_losses:
            current_lambda = adaptive_lambda_scheduler(
                epoch-1, epochs, 
                sum(epoch_source_losses)/len(epoch_source_losses),
                sum(epoch_target_losses)/len(epoch_target_losses),
                sum(epoch_domain_losses)/len(epoch_domain_losses),
                lambda_domain
            )
        else:
            current_lambda = max(0.001, lambda_domain * 0.01)
        
        epoch_stats = {'total_loss': 0, 'task_loss': 0, 'domain_loss': 0}
        epoch_source_losses, epoch_target_losses, epoch_domain_losses = [], [], []
        
        # ✅ 修复：分别迭代源域和目标域
        target_iter = iter(target_loader)
        n_batches = len(target_loader)
        
        progress_bar = tqdm(range(n_batches), desc=f'Epoch {epoch+1}/{epochs}', mininterval=3.0)
        
        for batch_idx in progress_bar:
            try:
                # 获取目标域批次
                try:
                    target_batch = next(target_iter)
                except StopIteration:
                    target_iter = iter(target_loader)
                    target_batch = next(target_iter)
                
                # 解析目标域数据
                if len(target_batch) == 4:
                    target_inputs, target_targets, _, target_category = target_batch
                else:
                    target_inputs, target_targets = target_batch[:2]
                    target_category = target_batch[3] if len(target_batch) > 3 else None
                
                target_inputs = target_inputs.float().to(device)
                target_targets = target_targets.float().to(device)
                if target_category is not None:
                    target_category = target_category.float().to(device)
                
                # ✅ 简化：只使用目标域数据进行微调（避免源域数据形状问题）
                if is_adaptive_model:
                    target_predictions = model(target_inputs, target_category, domain_idx=target_domain_idx)
                else:
                    target_predictions = model(target_inputs, target_category)
                
                # 任务损失
                target_task_loss = huber_loss_fn(target_predictions, target_targets)
                
                # 域对抗损失（简化版，仅使用目标域）
                domain_loss = torch.tensor(0.0, device=device)
                
                # 动态权重（随着训练增加目标域权重）
                tgt_weight = min(5.0, 1.0 + epoch*4/epochs)
                
                total_loss = target_task_loss * tgt_weight + domain_loss * current_lambda
                
                # 反向传播
                optimizer.zero_grad()
                total_loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                
                # 记录
                epoch_source_losses.append(0.0)  # 简化版不使用源域
                epoch_target_losses.append(target_task_loss.item())
                epoch_domain_losses.append(domain_loss.item())
                epoch_stats['total_loss'] += total_loss.item()
                
                progress_bar.set_postfix({
                    'loss': f"{total_loss.item():.4f}",
                    'tgt': f"{target_task_loss.item():.4f}"
                })
                
            except Exception as e:
                logging.warning(f"批次 {batch_idx} 处理出错: {e}")
                tb.print_exc()  # ✅ 使用别名
                continue
        
        scheduler.step()
        
        # 验证
        current_rmse = float('inf')
        with torch.no_grad():
            model.eval()
            val_preds, val_targets = [], []
            
            eval_batches = min(5, len(target_loader))
            eval_iter = iter(target_loader)
            
            for _ in range(eval_batches):
                try:
                    val_batch = next(eval_iter)
                    
                    if len(val_batch) == 4:
                        inputs, targets, _, category = val_batch
                    else:
                        inputs, targets = val_batch[:2]
                        category = val_batch[3] if len(val_batch) > 3 else None
                    
                    inputs = inputs.float().to(device)
                    targets = targets.float().to(device)
                    if category is not None:
                        category = category.float().to(device)
                    
                    if is_adaptive_model:
                        preds = model(inputs, category, domain_idx=target_domain_idx)
                    else:
                        preds = model(inputs, category)
                    
                    val_preds.append(preds)
                    val_targets.append(targets)
                except StopIteration:
                    break
                except Exception as e:
                    logging.warning(f"验证批次出错: {e}")
                    continue
            
            if val_preds:
                all_preds = torch.cat(val_preds, dim=0)
                all_tgts = torch.cat(val_targets, dim=0)
                current_rmse = torch.sqrt(torch.mean((all_preds - all_tgts) ** 2)).item()
                history['rmse'].append(current_rmse)
                
                if current_rmse < best_rmse:
                    best_rmse = current_rmse
                    best_model_state = copy.deepcopy(model.state_dict())
                    patience_counter = 0
                    logging.info(f"✅ 新最佳RMSE: {best_rmse:.4f}")
                else:
                    patience_counter += 1
        
        avg_loss = epoch_stats['total_loss'] / max(n_batches, 1)
        history['total_loss'].append(avg_loss)
        history['task_loss'].append(sum(epoch_target_losses)/max(len(epoch_target_losses), 1))
        history['domain_loss'].append(sum(epoch_domain_losses)/max(len(epoch_domain_losses), 1))
        
        logging.info(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f}, RMSE: {current_rmse:.4f}")
        
        if patience_counter >= early_stopping_patience:
            logging.info(f"早停于 epoch {epoch+1}")
            break
    
    if best_model_state:
        model.load_state_dict(best_model_state)
        logging.info(f"已恢复最佳模型 (RMSE: {best_rmse:.4f})")
    
    return model, history

In [4]:
def predict_with_uncertainty(model, inputs, category_onehot, domain_idx=None, mc_samples=100, device='cuda'):
    """
    使用MC Dropout进行不确定性估计 - 适配新的数据格式
    
    Args:
        model: 模型
        inputs: 输入特征 [batch, buildings, seq_len, features]
        category_onehot: 类别的one-hot编码 [batch, num_categories]
        domain_idx: 域索引（可选）
        mc_samples: MC采样次数
        device: 设备
    """
    model = model.to(device)
    inputs = inputs.to(device)
    category_onehot = category_onehot.to(device)  # 修改：从category改为category_onehot
    model.train()  # 开启dropout以进行随机采样
    
    predictions = []
    for _ in range(mc_samples):
        # 修改: 传入category_onehot而不是category
        outputs = model(inputs, category_onehot, domain_idx=domain_idx)
        
        if isinstance(outputs, tuple):
            outputs = outputs[0]  # 如果模型返回元组，取第一个元素
        
        predictions.append(outputs.detach())
    
    # 将预测堆叠为形状[mc_samples, batch, buildings, forecasts]
    try:
        stacked_preds = torch.stack(predictions, dim=0)
        
        # 计算平均值和标准差
        mean_pred = torch.mean(stacked_preds, dim=0)
        std_pred = torch.std(stacked_preds, dim=0)
        
        # 计算95%置信区间
        lower_bound = mean_pred - 1.96 * std_pred
        upper_bound = mean_pred + 1.96 * std_pred
        
        # 确保边界在有效范围内（注意：这里可能需要根据你的数据范围调整）
        # 如果数据已经归一化到[0,1]，保持不变
        # 如果没有归一化，需要调整为实际的数据范围
        lower_bound = torch.clamp(lower_bound, 0, float('inf'))  # 修改：电力消耗不能为负
        upper_bound = torch.clamp(upper_bound, 0, float('inf'))
        
        # 转换为CPU NumPy数组
        return (mean_pred.cpu().numpy(),
                lower_bound.cpu().numpy(),
                upper_bound.cpu().numpy(),
                std_pred.cpu().numpy())
    
    except Exception as e:
        print(f"处理MC采样结果时出错: {str(e)}")
        # 如果堆叠或其他操作失败，使用第一个样本作为预测
        try:
            mean_pred = predictions[0].cpu().numpy()
            std_pred = np.ones_like(mean_pred) * 0.1
            lower_bound = np.maximum(mean_pred - 1.96 * std_pred, 0)
            upper_bound = mean_pred + 1.96 * std_pred
            return mean_pred, lower_bound, upper_bound, std_pred
        except:
            # 最后的后备方案：返回零数组
            # 修改：根据输入形状推断输出形状
            if len(inputs.shape) == 4:  # [batch, buildings, seq_len, features]
                batch_size = inputs.shape[0]
                num_buildings = inputs.shape[1]
                output_shape = (batch_size, num_buildings, forecast_horizon)
            else:
                output_shape = (inputs.shape[0], forecast_horizon)
            
            zeros = np.zeros(output_shape)
            return zeros, zeros, zeros, zeros
def calculate_transfer_metrics(source_features, target_features, source_outputs=None, 
                              target_outputs=None, baseline_predictions=None, targets=None):
    """
    计算迁移学习的综合评估指标
    
    Args:
        source_features: 源域特征 [batch_size, feature_dim] 或 [batch_size, seq_len, feature_dim]
        target_features: 目标域特征 [batch_size, feature_dim] 或 [batch_size, seq_len, feature_dim]
        source_outputs: 源域模型输出（可选）
        target_outputs: 目标域模型输出（可选）
        baseline_predictions: 基线模型预测（可选）
        targets: 真实目标值（可选，用于计算负迁移）
    """
    import traceback  # 导入traceback以便打印详细错误信息
    
    # 处理输入特征的维度
    # 新的数据格式可能是 [batch, buildings, seq_len, features]
    if len(source_features.shape) == 4:  # [batch, buildings, seq_len, features]
        logging.info(f"检测到4D特征，进行展平: source_features.shape={source_features.shape}")
        # 先将buildings维度和batch维度合并，然后平均时间步
        batch_size, num_buildings, seq_len, feature_dim = source_features.shape
        source_features = source_features.reshape(batch_size * num_buildings, seq_len, feature_dim)
        source_features = source_features.mean(dim=1)  # [batch*buildings, feature_dim]
        
        # 对目标特征做同样处理
        target_features = target_features.reshape(
            target_features.shape[0] * target_features.shape[1], 
            target_features.shape[2], 
            target_features.shape[3]
        )
        target_features = target_features.mean(dim=1)
        
        logging.info(f"展平后: source_features.shape={source_features.shape}, target_features.shape={target_features.shape}")
    
    elif len(source_features.shape) == 3:  # [batch, seq_len, feature_dim]
        logging.info(f"检测到3D特征，进行展平: source_features.shape={source_features.shape}")
        # 平均所有时间步
        source_features = source_features.mean(dim=1)  # [batch, feature_dim]
        target_features = target_features.mean(dim=1)  # [batch, feature_dim]
        
        logging.info(f"展平后: source_features.shape={source_features.shape}, target_features.shape={target_features.shape}")
    
    # 确保特征维度匹配
    if source_features.size(-1) != target_features.size(-1):  # 使用-1获取最后一个维度
        logging.warning(f"特征维度不匹配: source_features.shape={source_features.shape}, target_features.shape={target_features.shape}")
        return {
            'a_distance': 'N/A',
            'feature_alignment': 'N/A',
            'mmd': 'N/A',
            'sample_efficiency': None
        }
    
    # 检查特征数量是否过大，可能导致内存问题
    max_samples = 5000
    if source_features.size(0) > max_samples or target_features.size(0) > max_samples:
        logging.warning(f"特征数量过大，进行采样: source_features.shape={source_features.shape}, target_features.shape={target_features.shape}")
        
        if source_features.size(0) > max_samples:
            indices = torch.randperm(source_features.size(0))[:max_samples]
            source_features = source_features[indices]
        
        if target_features.size(0) > max_samples:
            indices = torch.randperm(target_features.size(0))[:max_samples]
            target_features = target_features[indices]
        
        logging.info(f"采样后: source_features.shape={source_features.shape}, target_features.shape={target_features.shape}")
    
    # 下面的辅助函数保持不变
    def calculate_mmd(x, y):
        """计算最大平均差异(MMD)"""
        try:
            x_kernel = torch.mm(x, x.t())
            y_kernel = torch.mm(y, y.t())
            xy_kernel = torch.mm(x, y.t())
            return x_kernel.mean() + y_kernel.mean() - 2 * xy_kernel.mean()
        except Exception as e:
            logging.error(f"计算MMD时出错: {str(e)}")
            return float('nan')
    
    def calculate_a_distance(source_features, target_features):
        """计算A-distance"""
        try:
            domain_classifier = nn.Sequential(
                nn.Linear(source_features.size(1), 50),
                nn.ReLU(),
                nn.Linear(50, 1)
            ).to(source_features.device)
            
            source_domain_labels = torch.ones(source_features.size(0), 1).to(source_features.device)
            target_domain_labels = torch.zeros(target_features.size(0), 1).to(source_features.device)
            
            features = torch.cat([source_features, target_features], dim=0)
            labels = torch.cat([source_domain_labels, target_domain_labels], dim=0)
            
            optimizer = torch.optim.Adam(domain_classifier.parameters())
            criterion = nn.BCEWithLogitsLoss()
            
            for _ in range(100):
                optimizer.zero_grad()
                preds = domain_classifier(features)
                loss = criterion(preds, labels)
                loss.backward()
                optimizer.step()
            
            with torch.no_grad():
                preds = torch.sigmoid(domain_classifier(features))
                predicted_labels = (preds > 0.5).float()
                error = (predicted_labels != labels).float().mean()
            
            return 2 * (1 - 2 * error)
        except Exception as e:
            logging.error(f"计算A-distance时出错: {str(e)}")
            traceback.print_exc()
            return float('nan')
    
    def calculate_feature_alignment(source_features, target_features):
        """计算特征对齐质量"""
        try:
            source_norm = F.normalize(source_features, p=2, dim=1)
            target_norm = F.normalize(target_features, p=2, dim=1)
            similarity = torch.mm(source_norm, target_norm.t())
            return similarity.mean()
        except Exception as e:
            logging.error(f"计算特征对齐质量时出错: {str(e)}")
            return float('nan')
    
    def detect_negative_transfer(target_loss, baseline_loss):
        """检测负迁移"""
        if isinstance(target_loss, torch.Tensor):
            target_loss = target_loss.item()
        if isinstance(baseline_loss, torch.Tensor):
            baseline_loss = baseline_loss.item()
            
        is_negative = target_loss > baseline_loss
        transfer_gain = baseline_loss - target_loss
        
        return {
            'is_negative': bool(is_negative),
            'transfer_gain': float(transfer_gain)
        }
    
    def calculate_sample_efficiency(performance_curve):
        """计算样本效率"""
        target_performance = 0.9
        for i, perf in enumerate(performance_curve):
            if perf >= target_performance:
                return i + 1
        return len(performance_curve)

    # 计算基本指标
    try:
        metrics = {
            'a_distance': calculate_a_distance(source_features, target_features),
            'feature_alignment': calculate_feature_alignment(source_features, target_features),
            'mmd': calculate_mmd(source_features, target_features),
            'sample_efficiency': None
        }
    except Exception as e:
        logging.error(f"计算迁移学习指标时出错: {str(e)}")
        traceback.print_exc()
        return {
            'a_distance': 'N/A',
            'feature_alignment': 'N/A',
            'mmd': 'N/A',
            'sample_efficiency': None
        }
    
    # 如果提供了输出、基线预测和目标值，检测负迁移
    if target_outputs is not None and baseline_predictions is not None and targets is not None:
        try:
            # 处理可能的多维数据
            if isinstance(target_outputs, torch.Tensor):
                target_outputs = target_outputs.detach().cpu().numpy()
            if isinstance(baseline_predictions, torch.Tensor):
                baseline_predictions = baseline_predictions.detach().cpu().numpy()
            if isinstance(targets, torch.Tensor):
                targets = targets.detach().cpu().numpy()
            
            # 确保都是一维数组
            target_outputs = np.array(target_outputs).flatten()
            baseline_predictions = np.array(baseline_predictions).flatten()
            targets = np.array(targets).flatten()
            
            # 确保所有数组长度相同
            min_length = min(len(target_outputs), len(baseline_predictions), len(targets))
            target_outputs = target_outputs[:min_length]
            baseline_predictions = baseline_predictions[:min_length]
            targets = targets[:min_length]
            
            # 计算MSE损失
            target_loss = np.mean((target_outputs - targets) ** 2)
            baseline_loss = np.mean((baseline_predictions - targets) ** 2)
            
            logging.debug(f"target_loss: {target_loss}, type: {type(target_loss)}")
            logging.debug(f"baseline_loss: {baseline_loss}, type: {type(baseline_loss)}")
            
            # 检测负迁移
            neg_transfer = detect_negative_transfer(target_loss, baseline_loss)
            metrics['is_negative_transfer'] = neg_transfer['is_negative']
            metrics['transfer_gain'] = neg_transfer['transfer_gain']
            
        except Exception as e:
            logging.error(f"计算负迁移指标时出错: {str(e)}")
            traceback.print_exc()
            metrics['is_negative_transfer'] = 'N/A'
            metrics['transfer_gain'] = 'N/A'
    
    return metrics


import torch
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score
from calculate import calculate_metrics, print_metrics_table, calculate_uncertainty_metrics
def evaluate_model(model, test_loader, model_name="Model", baseline_model=None, 
                   device='cuda', domain_idx=None, num_categories=None):
    """
    评估模型在测试集上的性能（已修复格式化报错问题）
    """
    
    # 自动检测num_categories
    if num_categories is None:
        for batch in test_loader:
            if len(batch) >= 4:
                category_onehot = batch[3]
                num_categories = category_onehot.shape[-1]
                break
        if num_categories is None:
            num_categories = 6  # 默认值
    
    def calculate_improved_mape(y_true, y_pred, epsilon=0.01):
        mask = np.abs(y_true) > epsilon
        if not np.any(mask):
            return float('nan')
        return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

    # 辅助函数：安全打印指标
    def print_metric(name, value, unit="", fmt=".4f"):
        if isinstance(value, (int, float)):
            print(f"{name}: {value:{fmt}}{unit}")
        else:
            print(f"{name}: {value}")

    model.eval()
    all_predictions = []
    all_targets = []
    all_lower_bounds = []
    all_upper_bounds = []
    all_uncertainties = []
    source_features = []
    target_features = []
    source_outputs = []
    target_outputs = []
    
    # ===== ✅ 基线模型变量初始化 =====
    baseline_source_features = []
    baseline_target_features = []
    baseline_source_outputs = []
    baseline_target_outputs = []
    all_baseline_lower_bounds = []
    all_baseline_upper_bounds = []
    all_baseline_uncertainties = []
    # ================================================
    
    with torch.no_grad():
        for batch_data in tqdm(test_loader, desc=f"Evaluating {model_name}"):
            # 解包数据
            if len(batch_data) == 4:
                inputs, targets, category_name, category_onehot = batch_data
            else:
                inputs, targets = batch_data[:2]
                category_onehot = batch_data[3] if len(batch_data) > 3 else None
            
            inputs = inputs.float().to(device)
            targets = targets.float().to(device)
            
            if category_onehot is not None:
                category_onehot = category_onehot.float().to(device)
            else:
                batch_size = inputs.shape[0]
                category_onehot = torch.zeros(batch_size, num_categories).to(device)
                category_onehot[:, 0] = 1
            
            try:
                # ✅ 提取域特征
                if hasattr(model, 'time_series_encoder'):
                    try:
                        source_feat = model.time_series_encoder(inputs, domain_idx=0)
                        
                        # 处理不同维度的特征输出
                        if len(source_feat.shape) == 4:
                            source_feat = source_feat.mean(dim=2).reshape(-1, source_feat.shape[-1])
                        elif len(source_feat.shape) == 3:
                            source_feat = source_feat.mean(dim=1)
                        
                        source_features.append(source_feat.cpu())
                        
                        target_feat = model.time_series_encoder(inputs, domain_idx=1)
                        if len(target_feat.shape) == 4:
                            target_feat = target_feat.mean(dim=2).reshape(-1, target_feat.shape[-1])
                        elif len(target_feat.shape) == 3:
                            target_feat = target_feat.mean(dim=1)
                        
                        target_features.append(target_feat.cpu())
                        
                    except Exception as e:
                        pass # 静默处理特征提取错误
                
                # 不确定性预测
                mean_pred, lower_bound, upper_bound, uncertainty = predict_with_uncertainty(
                    model=model, 
                    inputs=inputs, 
                    category_onehot=category_onehot,
                    domain_idx=domain_idx, 
                    device=device, 
                    mc_samples=50
                )
                
                if isinstance(mean_pred, torch.Tensor):
                    mean_pred = mean_pred.cpu().numpy()
                
                # 收集域输出
                if hasattr(model, 'time_series_encoder'):
                    source_outputs.append(mean_pred)
                    try:
                        target_pred = model(inputs, category_onehot, domain_idx=domain_idx)
                        if isinstance(target_pred, torch.Tensor):
                            target_pred = target_pred.cpu().numpy()
                        target_outputs.append(target_pred)
                    except:
                        target_outputs.append(mean_pred)
                
                all_predictions.append(mean_pred)
                all_targets.append(targets.cpu().numpy())
                all_lower_bounds.append(lower_bound)
                all_upper_bounds.append(upper_bound)
                all_uncertainties.append(uncertainty)
                
            except Exception as e:
                print(f"批次处理错误: {str(e)}")
                continue
            
            # 处理基线模型
            if baseline_model is not None:
                try:
                    if hasattr(baseline_model, 'time_series_encoder'):
                        try:
                            # 处理4D输入
                            if len(inputs.shape) == 4:
                                batch_size, num_buildings, seq_len, features = inputs.shape
                                inputs_reshaped = inputs.reshape(batch_size * num_buildings, seq_len, features)
                                baseline_source_feat = baseline_model.time_series_encoder(inputs_reshaped, domain_idx=0)
                                baseline_source_features.append(baseline_source_feat.mean(dim=1))
                                baseline_target_feat = baseline_model.time_series_encoder(inputs_reshaped, domain_idx=1)
                                baseline_target_features.append(baseline_target_feat.mean(dim=1))
                            else:
                                baseline_source_feat = baseline_model.time_series_encoder(inputs, domain_idx=0)
                                baseline_source_features.append(baseline_source_feat.mean(dim=1))
                                baseline_target_feat = baseline_model.time_series_encoder(inputs, domain_idx=1)
                                baseline_target_features.append(baseline_target_feat.mean(dim=1))
                        except Exception as e:
                            pass
                    
                    if hasattr(baseline_model, 'predict'):
                        baseline_pred = baseline_model.predict(inputs)
                    else:
                        baseline_pred = baseline_model(inputs, category_onehot)
                        if isinstance(baseline_pred, tuple):
                            baseline_pred = baseline_pred[0]
                    
                    if isinstance(baseline_pred, torch.Tensor):
                        baseline_pred = baseline_pred.cpu().numpy()
                    
                    baseline_source_outputs.append(baseline_pred)
                    baseline_target_outputs.append(baseline_pred)
                    
                    # 处理基线模型不确定性
                    try:
                        if hasattr(baseline_model, 'predict_with_uncertainty'):
                            bl_mean, bl_lower, bl_upper, bl_uncertainty = predict_with_uncertainty(
                                model=baseline_model, 
                                inputs=inputs, 
                                category_onehot=category_onehot,
                                domain_idx=domain_idx, 
                                device=device, 
                                mc_samples=50
                            )
                            all_baseline_lower_bounds.append(bl_lower)
                            all_baseline_upper_bounds.append(bl_upper)
                            all_baseline_uncertainties.append(bl_uncertainty)
                        else:
                            bl_pred = baseline_pred.reshape(-1, 1)
                            all_baseline_lower_bounds.append(bl_pred * 0.9)
                            all_baseline_upper_bounds.append(bl_pred * 1.1)
                            all_baseline_uncertainties.append(np.ones_like(bl_pred) * 0.1)
                    except Exception as e:
                        bl_pred = baseline_pred.reshape(-1, 1)
                        all_baseline_lower_bounds.append(bl_pred * 0.9)
                        all_baseline_upper_bounds.append(bl_pred * 1.1)
                        all_baseline_uncertainties.append(np.ones_like(bl_pred) * 0.1)
                        
                except Exception as e:
                    print(f"处理基线模型时出错: {str(e)}")
    
    if not all_predictions:
        print("警告: 无有效预测，无法评估")
        return {"Model": model_name, "Error": "无有效预测"}
    
    # 合并数据
    try:
        if isinstance(all_predictions[0], np.ndarray) and len(all_predictions[0].shape) > 2:
            all_predictions = [p.reshape(-1, p.shape[-1]) if len(p.shape) > 2 else p for p in all_predictions]
            all_targets = [t.reshape(-1, t.shape[-1]) if len(t.shape) > 2 else t for t in all_targets]
        
        all_predictions = np.concatenate(all_predictions, axis=0)
        all_targets = np.concatenate(all_targets, axis=0)
    except Exception as e:
        return {"Model": model_name, "Error": f"合并失败: {str(e)}"}
    
    # 处理不确定性合并
    if not all_lower_bounds or not all_upper_bounds:
        all_lower_bounds = all_predictions * 0.9
        all_upper_bounds = all_predictions * 1.1
        all_uncertainties = np.ones_like(all_predictions) * 0.1
    else:
        try:
            if isinstance(all_lower_bounds[0], np.ndarray) and len(all_lower_bounds[0].shape) > 2:
                all_lower_bounds = [b.reshape(-1, b.shape[-1]) if len(b.shape) > 2 else b for b in all_lower_bounds]
                all_upper_bounds = [b.reshape(-1, b.shape[-1]) if len(b.shape) > 2 else b for b in all_upper_bounds]
                all_uncertainties = [u.reshape(-1, u.shape[-1]) if len(u.shape) > 2 else u for u in all_uncertainties]
            
            all_lower_bounds = np.concatenate(all_lower_bounds, axis=0)
            all_upper_bounds = np.concatenate(all_upper_bounds, axis=0)
            all_uncertainties = np.concatenate(all_uncertainties, axis=0)
        except:
            all_lower_bounds = all_predictions * 0.9
            all_upper_bounds = all_predictions * 1.1
            all_uncertainties = np.ones_like(all_predictions) * 0.1
    
    all_predictions_flat = all_predictions.reshape(-1)
    all_targets_flat = all_targets.reshape(-1)
    
    # 对齐形状
    if all_lower_bounds.size != all_predictions_flat.size:
        min_size = min(all_lower_bounds.size, all_predictions_flat.size)
        all_lower_bounds = all_lower_bounds.reshape(-1)[:min_size]
        all_upper_bounds = all_upper_bounds.reshape(-1)[:min_size]
        all_uncertainties = all_uncertainties.reshape(-1)[:min_size]
        all_predictions_flat = all_predictions_flat[:min_size]
        all_targets_flat = all_targets_flat[:min_size]
    else:
        all_lower_bounds = all_lower_bounds.reshape(-1)
        all_upper_bounds = all_upper_bounds.reshape(-1)
        all_uncertainties = all_uncertainties.reshape(-1)
    
    # 处理基线预测聚合
    baseline_predictions = None
    baseline_metrics = {}
    if baseline_model is not None:
        try:
            if baseline_source_outputs:
                 # 注意：这里我们使用在循环中收集的 baseline_source_outputs，避免重复预测
                baseline_predictions = np.concatenate(baseline_source_outputs, axis=0).reshape(-1)
                
                if baseline_predictions.size != all_predictions_flat.size:
                    baseline_predictions = baseline_predictions[:all_predictions_flat.size]
                
                # 合并基线不确定性
                if all_baseline_lower_bounds:
                    try:
                        all_baseline_lower_bounds = np.concatenate(all_baseline_lower_bounds, axis=0).reshape(-1)
                        all_baseline_upper_bounds = np.concatenate(all_baseline_upper_bounds, axis=0).reshape(-1)
                        all_baseline_uncertainties = np.concatenate(all_baseline_uncertainties, axis=0).reshape(-1)
                        
                        if all_baseline_lower_bounds.size != baseline_predictions.size:
                            min_len = min(all_baseline_lower_bounds.size, baseline_predictions.size)
                            all_baseline_lower_bounds = all_baseline_lower_bounds[:min_len]
                            all_baseline_upper_bounds = all_baseline_upper_bounds[:min_len]
                            all_baseline_uncertainties = all_baseline_uncertainties[:min_len]
                    except:
                        all_baseline_lower_bounds = baseline_predictions * 0.9
                        all_baseline_upper_bounds = baseline_predictions * 1.1
                        all_baseline_uncertainties = np.ones_like(baseline_predictions) * 0.1
                else:
                    all_baseline_lower_bounds = baseline_predictions * 0.9
                    all_baseline_upper_bounds = baseline_predictions * 1.1
                    all_baseline_uncertainties = np.ones_like(baseline_predictions) * 0.1

                baseline_metrics = calculate_metrics(
                    predictions=baseline_predictions,
                    real_values=all_targets_flat,
                    model_name=f"Baseline",
                    baseline_predictions=None,
                    lower_bounds=all_baseline_lower_bounds,
                    upper_bounds=all_baseline_upper_bounds,
                    confidence=0.95
                )
                baseline_metrics['MAPE'] = calculate_improved_mape(all_targets_flat, baseline_predictions, epsilon=0.01)
                baseline_metrics['avg_uncertainty'] = np.mean(all_baseline_uncertainties)
        except Exception as e:
            print(f"处理基线数据错误: {str(e)}")
    
    # 计算主模型指标
    try:
        metrics = calculate_metrics(
            predictions=all_predictions_flat,
            real_values=all_targets_flat,
            model_name=model_name,
            baseline_predictions=baseline_predictions,
            lower_bounds=all_lower_bounds,
            upper_bounds=all_upper_bounds,
            confidence=0.95
        )
        
        metrics['avg_uncertainty'] = np.mean(all_uncertainties)
        metrics['MAPE'] = calculate_improved_mape(all_targets_flat, all_predictions_flat, epsilon=0.01)
    
        # 迁移学习指标
        if source_features and target_features:
            try:
                source_features_tensor = torch.cat(source_features, dim=0).cpu()
                target_features_tensor = torch.cat(target_features, dim=0).cpu()
                source_outputs_arr = np.concatenate(source_outputs, axis=0)
                target_outputs_arr = np.concatenate(target_outputs, axis=0)
                
                transfer_metrics = calculate_transfer_metrics(
                    source_features=source_features_tensor,
                    target_features=target_features_tensor,
                    source_outputs=source_outputs_arr,
                    target_outputs=target_outputs_arr,
                    baseline_predictions=baseline_predictions,
                    targets=all_targets_flat
                )
                metrics.update(transfer_metrics)
                
                # 基线迁移指标
                if baseline_source_features and baseline_target_features:
                    try:
                        baseline_source_features_tensor = torch.cat(baseline_source_features, dim=0).cpu()
                        baseline_target_features_tensor = torch.cat(baseline_target_features, dim=0).cpu()
                        baseline_source_outputs_arr = np.concatenate(baseline_source_outputs, axis=0)
                        baseline_target_outputs_arr = np.concatenate(baseline_target_outputs, axis=0)
                        
                        baseline_transfer_metrics = calculate_transfer_metrics(
                            source_features=baseline_source_features_tensor,
                            target_features=baseline_target_features_tensor,
                            source_outputs=baseline_source_outputs_arr,
                            target_outputs=baseline_target_outputs_arr,
                            baseline_predictions=baseline_predictions,
                            targets=all_targets_flat
                        )
                        
                        baseline_transfer_metrics = {f"baseline_{k}": v for k, v in baseline_transfer_metrics.items()}
                        metrics.update(baseline_transfer_metrics)
                        
                        # 计算改进率
                        if 'a_distance' in transfer_metrics and 'baseline_a_distance' in metrics:
                            metrics['a_distance_improvement'] = metrics['baseline_a_distance'] - metrics['a_distance']
                    except Exception as e:
                        print(f"基线迁移指标错误: {e}")
            except Exception as e:
                print(f"迁移指标错误: {e}")
    
    except Exception as e:
        print(f"指标计算严重错误: {str(e)}")
        # 兜底
        rmse = np.sqrt(mean_squared_error(all_targets_flat, all_predictions_flat))
        metrics = {'Model': model_name, 'RMSD': rmse}

    # ===== ✅ 打印报告（已修复报错部分） =====
    print(f"\n{model_name} 评估报告:")
    print_metric("RMSD", metrics.get('RMSD', 'N/A'))
    print_metric("MAPE", metrics.get('MAPE', 'N/A'), unit="%", fmt=".2f")
    print_metric("R²", metrics.get('R2', 'N/A'))
    print_metric("CV-RMSE", metrics.get('CV-RMSE', 'N/A'), unit="%") # 修复点
    print_metric("SD_real", metrics.get('SD_real', 'N/A'))
    print_metric("SD_pred", metrics.get('SD_pred', 'N/A'))
    print_metric("CC", metrics.get('CC', 'N/A'))
    
    if isinstance(metrics.get('PIR'), (int, float)):
        print(f"RMSD改进率(PIR): {metrics['PIR']:.2f}%")
    
    if baseline_metrics:
        print(f"\nBaseline Model 评估报告:")
        print_metric("RMSD", baseline_metrics.get('RMSD', 'N/A'))
        print_metric("MAPE", baseline_metrics.get('MAPE', 'N/A'), unit="%", fmt=".2f")
        print_metric("R²", baseline_metrics.get('R2', 'N/A'))
        print_metric("CV-RMSE", baseline_metrics.get('CV-RMSE', 'N/A'), unit="%") # 修复点
        print_metric("SD_real", baseline_metrics.get('SD_real', 'N/A'))
        print_metric("SD_pred", baseline_metrics.get('SD_pred', 'N/A'))
        print_metric("CC", baseline_metrics.get('CC', 'N/A'))
        
        if 'PICP' in baseline_metrics:
            print("\n基线模型不确定性评估:")
            print_metric("PICP", baseline_metrics.get('PICP', 'N/A'), unit="%", fmt=".2f")
            print_metric("NMPIW", baseline_metrics.get('NMPIW', 'N/A'))
            print_metric("校准误差", baseline_metrics.get('calibration_error', 'N/A'), unit="%", fmt=".2f")
            print_metric("平均不确定性", baseline_metrics.get('avg_uncertainty', 'N/A'))
    
    # 打印迁移学习评估对比
    if any(key in metrics for key in ['a_distance', 'feature_alignment', 'mmd']):
        print("\n迁移学习评估:")
        print_metric("A-distance", metrics.get('a_distance', 'N/A'))
        print_metric("特征对齐", metrics.get('feature_alignment', 'N/A'))
        print_metric("MMD", metrics.get('mmd', 'N/A'))
    
    if 'PICP' in metrics:
        print("\n不确定性评估:")
        print_metric("PICP", metrics.get('PICP', 'N/A'), unit="%", fmt=".2f")
        print_metric("NMPIW", metrics.get('NMPIW', 'N/A'))
        print_metric("校准误差", metrics.get('calibration_error', 'N/A'), unit="%", fmt=".2f")
        print_metric("UQS", metrics.get('UQS', 'N/A'))
        print_metric("平均不确定性", metrics.get('avg_uncertainty', 'N/A'))
    

    


    
    try:
        sample_idx = 0
        if len(all_predictions.shape) == 3 and all_predictions.shape[0] > 0 and all_predictions.shape[1] > 0 and all_predictions.shape[2] > 0:
            max_horizon = min(10, all_predictions.shape[2])
            plt.figure(figsize=(10, 6))
            try:
                plt.fill_between(
                    range(max_horizon),
                    all_lower_bounds.reshape(all_predictions.shape)[sample_idx, 0, :max_horizon],
                    all_upper_bounds.reshape(all_predictions.shape)[sample_idx, 0, :max_horizon],
                    alpha=0.3, color='blue', label='95% Confidence Interval'
                )
                plt.plot(range(max_horizon), all_predictions[sample_idx, 0, :max_horizon], 'b-o', label='Predictions')
                plt.plot(range(max_horizon), all_targets[sample_idx, 0, :max_horizon], 'r-x', label='Ground Truth')
            except Exception as e:
                print(f"3D data visualization error: {str(e)}")
                plt.plot(range(max_horizon), all_predictions[sample_idx, 0, :max_horizon], 'b-o', label='Predictions')
                plt.plot(range(max_horizon), all_targets[sample_idx, 0, :max_horizon], 'r-x', label='Ground Truth')
        
        elif len(all_predictions.shape) == 2 and all_predictions.shape[0] > 0 and all_predictions.shape[1] > 0:
            max_horizon = min(10, all_predictions.shape[1])
            plt.figure(figsize=(10, 6))
            try:
                plt.fill_between(
                    range(max_horizon),
                    all_lower_bounds.reshape(all_predictions.shape)[sample_idx, :max_horizon],
                    all_upper_bounds.reshape(all_predictions.shape)[sample_idx, :max_horizon],
                    alpha=0.3, color='blue', label='95% Confidence Interval'
                )
                plt.plot(range(max_horizon), all_predictions[sample_idx, :max_horizon], 'b-o', label='Predictions')
                plt.plot(range(max_horizon), all_targets[sample_idx, :max_horizon], 'r-x', label='Ground Truth')
            except Exception as e:
                print(f"2D data visualization error: {str(e)}")
                plt.plot(range(max_horizon), all_predictions[sample_idx, :max_horizon], 'b-o', label='Predictions')
                plt.plot(range(max_horizon), all_targets[sample_idx, :max_horizon], 'r-x', label='Ground Truth')
        
        elif len(all_predictions.shape) == 1 and all_predictions.shape[0] > 0:
            max_points = min(10, all_predictions.shape[0])
            plt.figure(figsize=(10, 6))
            try:
                plt.fill_between(
                    range(max_points),
                    all_lower_bounds[:max_points],
                    all_upper_bounds[:max_points],
                    alpha=0.3, color='blue', label='95% Confidence Interval'
                )
                plt.plot(range(max_points), all_predictions[:max_points], 'b-o', label='Predictions')
                plt.plot(range(max_points), all_targets[:max_points], 'r-x', label='Ground Truth')
            except Exception as e:
                print(f"1D data visualization error: {str(e)}")
                plt.plot(range(max_points), all_predictions[:max_points], 'b-o', label='Predictions')
                plt.plot(range(max_points), all_targets[:max_points], 'r-x', label='Ground Truth')
        
        else:
            print(f"Cannot visualize predictions: incompatible shape {all_predictions.shape}")
            return metrics
        
        plt.title(f'{model_name} Prediction Example\nPICP: {metrics.get("PICP", "N/A")}%')
        plt.xlabel('Prediction Time Step')
        plt.ylabel('Value')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        safe_model_name = ''.join(c if c.isalnum() or c in ['-', '_'] else '_' for c in model_name)
        plt.savefig(f'prediction_sample_{safe_model_name}.png')
        plt.close()
    except Exception as e:
        print(f"可视化错误: {str(e)}")
        print(f"数据形状: 预测={all_predictions.shape}, 目标={all_targets.shape}")
    
    if baseline_metrics:
        metrics['baseline_metrics'] = baseline_metrics
    
    return metrics

In [5]:
import os
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import logging
import traceback
import copy
from datetime import datetime
from torch.optim import AdamW
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm

# 1. 导入基础设施
from dataloader import (
    load_source_domain_dataloaders,
    load_transfer_learning_dataloaders
)
# 导入原有的预测器组件
from AdaptiveBILSTM import BiLSTMPredictor
# 导入训练相关函数 (假设在 training.py 中)

# ==========================================
# 🔥 1. 定义实验标签 & 配置
# ==========================================
EXPERIMENT_TAG = "NoFeature"  # 实验标签
DATA_SHORTAGE_SCENARIOS = ['mild', 'heavy', 'extreme']
# 🔧 调试模式：仅测试 DO 和 CC
TEST_CATEGORIES = ["DO", "CC"] 

# 配置日志
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(f'training_{EXPERIMENT_TAG}.log', encoding='utf-8'),
        logging.StreamHandler()
    ]
)

# 参数设置
BATCH_SIZE = 32
SEQUENCE_LENGTH = 24
FORECAST_HORIZON = 24
HIDDEN_DIM = 64
NUM_LAYERS = 2
DROPOUT = 0.2
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4
EPOCHS = 10
INPUT_DIM = 6 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 启动消融实验: {EXPERIMENT_TAG} | 设备: {device}")

# ==========================================
# 2. 定义消融模型 (NoFeatureExtraction)
# ==========================================

class TimeSeriesEncoder_NoFeature(nn.Module):
    """
    消融版编码器：
    - 移除 MultiScaleEncoder
    - 移除 TemporalDependency
    - 保留 Gaussian Domain Adaptation (贝叶斯域适应)
    """
    def __init__(self, input_dim, hidden_dim, num_domains=7, num_layers=2, dropout=0.3):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = (hidden_dim // 4) * 4
        self.num_domains = num_domains
        
        # 1. 简单的特征投影 (替代复杂特征提取)
        self.simple_feature_projection = nn.Sequential(
            nn.Linear(input_dim, self.hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.LayerNorm(self.hidden_dim)
        )
        
        # 2. 贝叶斯域自适应参数 (保留)
        self.domain_mu = nn.Parameter(torch.zeros(num_domains, self.hidden_dim))
        self.domain_logvar = nn.Parameter(torch.zeros(num_domains, self.hidden_dim))
        self.domain_importance = nn.Parameter(torch.ones(num_domains))
        
        # 3. 域适应基础网络
        self.domain_adapter_base = nn.Sequential(
            nn.Linear(self.hidden_dim, self.hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
    
    def gaussian_domain_adapt(self, features, domain_idx=None):
        if domain_idx is not None:
            mu = self.domain_mu[domain_idx]
            logvar = self.domain_logvar[domain_idx]
            importance = F.softplus(self.domain_importance[domain_idx])
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            domain_params = mu + eps * std
            adapted_features = features * (domain_params * importance.unsqueeze(-1))
        else:
            domain_weights = F.softmax(self.domain_importance, dim=0)
            mixed_mu = torch.sum(self.domain_mu * domain_weights.unsqueeze(1), dim=0)
            mixed_logvar = torch.sum(self.domain_logvar * domain_weights.unsqueeze(1), dim=0)
            std = torch.exp(0.5 * mixed_logvar)
            eps = torch.randn_like(std)
            domain_params = mixed_mu + eps * std
            adapted_features = features * domain_params
        
        return self.domain_adapter_base(adapted_features)
    
    def forward(self, x, domain_idx=None):
        batch_size, num_buildings, seq_len, _ = x.shape
        x_reshaped = x.view(batch_size * num_buildings, seq_len, self.input_dim)
        
        # 简单投影
        base_features = self.simple_feature_projection(x_reshaped)
        
        # 贝叶斯域适应
        adapted_features = []
        for t in range(seq_len):
            t_feat = base_features[:, t, :]
            t_adapted = self.gaussian_domain_adapt(t_feat, domain_idx)
            adapted_features.append(t_adapted)
        adapted_features = torch.stack(adapted_features, dim=1)
    
        return adapted_features.view(batch_size, num_buildings, seq_len, self.hidden_dim)

class AdaptiveBiLSTM_NoFeatureExtraction(nn.Module):
    def __init__(self, input_dim, hidden_dim, category_dim, forecast_horizon, 
                 num_buildings, num_domains=7, num_layers=2, dropout=0.3):
        super().__init__()
        
        self.time_series_encoder = TimeSeriesEncoder_NoFeature(
            input_dim=input_dim,
            hidden_dim=hidden_dim,
            num_domains=num_domains,
            num_layers=num_layers,
            dropout=dropout
        )
        
        # 复用原有的预测器
        self.bilstm_predictor = BiLSTMPredictor(
            input_dim=hidden_dim,
            hidden_dim=hidden_dim,
            category_dim=category_dim,
            forecast_horizon=forecast_horizon,
            num_buildings=num_buildings,
            num_layers=num_layers,
            dropout=dropout
        )
    
    def forward(self, x, category, domain_idx=None):
        time_features = self.time_series_encoder(x, domain_idx)
        predictions = self.bilstm_predictor(time_features, category)
        return predictions

# ==========================================
# 3. 源域模型训练 (Source Domain Training)
# ==========================================
print(f"\n[{EXPERIMENT_TAG}] 1. 准备源域模型...")

# 加载源域数据
train_loader, val_loader, all_categories = load_source_domain_dataloaders(
    batch_size=BATCH_SIZE, sequence_length=SEQUENCE_LENGTH,
    forecast_horizon=FORECAST_HORIZON, handle_missing='forward_fill', val_ratio=0.2
)

# 过滤类别 (调试用)
# categories = [c for c in all_categories if c in TEST_CATEGORIES]
# 如果你想跑全量，请注释掉上面这行，改用下面这行：
categories = list(all_categories)
print(f"🔧 测试类别范围: {categories}")

# 定义保存路径 (带标签)
source_model_path = f'models/source_{EXPERIMENT_TAG}_best.pth'

# 实例化消融模型
source_model = AdaptiveBiLSTM_NoFeatureExtraction(
    input_dim=INPUT_DIM,
    hidden_dim=HIDDEN_DIM,
    category_dim=len(all_categories), # 必须对应原始总类别数
    forecast_horizon=FORECAST_HORIZON,
    num_buildings=1,
    num_domains=len(all_categories),
    num_layers=NUM_LAYERS,
    dropout=DROPOUT
).to(device)

# 检查是否存在
if os.path.exists(source_model_path):
    print(f"✅ 加载已存在的消融源域模型: {source_model_path}")
    checkpoint = torch.load(source_model_path, weights_only=False)
    if isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
        source_model.load_state_dict(checkpoint['state_dict'])
    else:
        source_model.load_state_dict(checkpoint)
else:
    print(f"⚠️ 未找到源域模型，开始从头训练 {EXPERIMENT_TAG} ...")
    source_model, _, _ = train_and_save_model(
        model=source_model,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=EPOCHS,
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        model_name=f'source_{EXPERIMENT_TAG}', # 文件名带标签
        save_dir='models',
        device=device,
        source_domain_idx=0,
        target_domain_idx=0
    )
    print("✅ 源域模型训练完成。")

# ==========================================
# 4. 迁移学习实验 (DANN on NoFeature)
# ==========================================
print(f"\n[{EXPERIMENT_TAG}] 2. 开始迁移学习实验...")

DOMAIN_MAPPING = {"DO": 1, "HO": 2, "LI": 3, "OF": 4, "UL": 5, "CC": 0}
transfer_results = {}

for target_category in categories:
    print(f"\n---> 当前处理类别: {target_category}")
    transfer_results[target_category] = {}
    target_domain_idx = DOMAIN_MAPPING.get(target_category, 0)
    
    for data_shortage in DATA_SHORTAGE_SCENARIOS:
        logging.info(f"\n Source({EXPERIMENT_TAG}) ➜ Target({target_category}), Shortage: {data_shortage}")
        
        try:
            # 加载数据
            tl_train_loader, tl_test_loader, _ = load_transfer_learning_dataloaders(
                category=target_category,
                data_shortage=data_shortage,
                batch_size=BATCH_SIZE,
                sequence_length=SEQUENCE_LENGTH,
                forecast_horizon=FORECAST_HORIZON,
                handle_missing='forward_fill'
            )
            
            if tl_train_loader is None: continue
            
            # ✅ 执行 DANN 迁移
            logging.info(f" 开始 DANN 适应...")
            
            adapted_model, transfer_history = adapt_to_target_domain(
                source_model=source_model, # 消融版模型
                source_loader=tl_train_loader,
                target_loader=tl_test_loader,
                epochs=10, 
                lr=LEARNING_RATE / 2,
                device=device,
                lambda_domain=0.4, 
                source_domain_idx=0,
                target_domain_idx=target_domain_idx
            )
            
            if adapted_model is None: continue

            # ✅ 评估模型 (不带 Baseline)
            logging.info(f" 评估...")
            metrics = evaluate_model(
                model=adapted_model,
                test_loader=tl_test_loader,
                model_name=f"{EXPERIMENT_TAG}_TL_{target_category}_{data_shortage}",
                baseline_model=None, # 🚫 禁用 Baseline
                device=device,
                domain_idx=target_domain_idx
            )

            # 保存结果结构
            model_type = f"adaptive_{EXPERIMENT_TAG.lower()}"
            
            transfer_results[target_category][data_shortage] = {
                'metrics': metrics,
                'model_type': model_type,
                'transfer_metrics': metrics.get('transfer_metrics', {}),
                'evaluation_metrics': metrics
            }

            # 保存模型权重 (带标签)
            model_save_path = f'models/{model_type}_TL_{target_category}_{data_shortage}.pth'
            torch.save(adapted_model.state_dict(), model_save_path)
            logging.info(f" 模型已保存到 {model_save_path}")
            
            print(f"   Result: RMSD={metrics.get('RMSD', 'N/A'):.4f}, MAPE={metrics.get('MAPE', 'N/A'):.2f}%")

        except Exception as e:
            logging.error(f"[ERROR] 错误: {target_category}/{data_shortage}: {e}")
            traceback.print_exc()
            continue

# ==========================================
# 5. 保存最终结果 CSV
# ==========================================
import csv

def convert_to_basic_type(obj):
    if isinstance(obj, (np.ndarray, torch.Tensor)):
        if obj.size == 1: return float(obj.item())
        return str(obj.tolist())
    elif isinstance(obj, (np.float32, np.float64)): return float(obj)
    return str(obj) if obj is not None else "N/A"

csv_path = f"models/results_{EXPERIMENT_TAG}_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
csv_header = [
    "Category", "Shortage", "Model", 
    "RMSD", "MAPE", "R2", "CV-RMSE", 
    "PICP", "NMPIW", "CE",
    "A-distance", "MMD"
]
csv_rows = []

for cat, shortages in transfer_results.items():
    for shortage, res in shortages.items():
        m = res['metrics']
        t = res.get('transfer_metrics', {})
        if not t: t = m
            
        csv_rows.append([
            cat, shortage, f"Adaptive_{EXPERIMENT_TAG}",
            convert_to_basic_type(m.get('RMSD')), 
            convert_to_basic_type(m.get('MAPE')), 
            convert_to_basic_type(m.get('R2')), 
            convert_to_basic_type(m.get('CV-RMSE')),
            convert_to_basic_type(m.get('PICP')),
            convert_to_basic_type(m.get('NMPIW')),
            convert_to_basic_type(m.get('calibration_error')),
            convert_to_basic_type(t.get('a_distance')),
            convert_to_basic_type(t.get('mmd'))
        ])

try:
    with open(csv_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(csv_header)
        writer.writerows(csv_rows)
    print(f"\n✅ 消融实验完成！结果已保存至 {csv_path}")
except Exception as e:
    print(f"保存 CSV 失败: {e}")
    # 备选：保存 JSON
    with open(csv_path.replace('.csv', '.json'), 'w') as f:
        def default(o): return str(o)
        json.dump(transfer_results, f, indent=4, default=default)

🚀 启动消融实验: NoFeature | 设备: cuda

[NoFeature] 1. 准备源域模型...
📊 加载源域训练数据
说明:
  • 仅使用train_test_labels.json中标记为train的建筑
  • 使用完整数据（无人为引入的缺失）
  • 按时间划分: 前80%用于训练, 后20%用于验证

🏢 建筑分组:
  训练建筑: 25 个

⚡ 加载电力数据...
  ✓ 完整电力数据: (17544, 25)
  ✓ 训练集电力: (17544, 25)

🌤️  加载天气数据...
  训练集天气数据:
    ✓ Hog: (17542, 5)
    ✓ Rat: (17539, 5)
    ✓ Robin: (17516, 5)
    ✓ Wolf: (17505, 5)
    ✓ Eagle: (17536, 5)

🔄 创建数据加载器...
  数据集: 25 个建筑, 13938 个有效样本 (first_80_percent)


2026-01-14 16:59:43,954 - INFO - 
 Source(NoFeature) ➜ Target(DO), Shortage: mild


  数据集: 25 个建筑, 3449 个有效样本 (last_20_percent)

✅ 源域数据加载完成!
训练样本: 13938 (前80%时间)
验证样本: 3449 (后20%时间)
批次大小: 32
缺失处理: forward_fill

🔧 测试类别范围: ['DO', 'HO', 'LI', 'OF', 'UL', 'CC']
✅ 加载已存在的消融源域模型: models/source_NoFeature_best.pth

[NoFeature] 2. 开始迁移学习实验...

---> 当前处理类别: DO
📊 加载迁移学习数据 - 类别: DO, 稀缺度: MILD
说明:
  • 训练数据: train标签建筑(完整) + 目标建筑稀缺训练集(前80%)
  • 测试数据: 目标建筑稀缺测试集(后20%)

🏢 建筑分组:
  训练建筑: 5 个 - ['Hog_lodging_Brian', 'Hog_lodging_Nikki', 'Hog_lodging_Ora', 'Robin_lodging_Celia', 'Robin_lodging_Elmer']
  目标建筑: 1 个 - ['Robin_lodging_Renea']

⚡ 加载电力数据...
  ✓ 完整电力数据: (17544, 25)
  ✓ 稀缺电力数据: (17544, 6)
  ✓ 稀缺训练集: (14035, 6) (前80%)
  ✓ 稀缺测试集: (3509, 6) (后20%)

🌤️  加载天气数据...
  训练建筑天气数据 (完整):
    ✓ Robin: (17516, 5)
    ✓ Hog: (17542, 5)
  目标建筑天气数据 (mild):
    ✓ Robin: (17516, 5) (缺失率: 16.02%)

🔄 创建数据加载器...
  创建训练建筑数据集 (5 个建筑)...


2026-01-14 16:59:44,434 - INFO -  开始 DANN 适应...


  数据集: 5 个建筑, 17462 个有效样本
  创建目标建筑训练集数据集 (1 个建筑)...
  ⚠️  数据包含缺失 (缺失率: 20.00%)
  数据集: 1 个建筑, 13962 个有效样本
  创建目标建筑测试集数据集 (1 个建筑)...
  数据集: 1 个建筑, 3456 个有效样本

✅ 类别 DO 迁移学习数据加载完成!
训练样本: 31424
  - 训练建筑数据: 17462 样本
  - 目标建筑训练数据: 13962 样本
测试样本: 3456
批次大小: 32
缺失处理: forward_fill



...

   Result: RMSD=0.1395, MAPE=37.90%
📊 加载迁移学习数据 - 类别: CC, 稀缺度: EXTREME
说明:
  • 训练数据: train标签建筑(完整) + 目标建筑稀缺训练集(前80%)
  • 测试数据: 目标建筑稀缺测试集(后20%)
  • 特殊处理: CC类别无训练建筑，从其他5类各随机选1个建筑
  从类别 DO 随机选择建筑: Hog_lodging_Nikki
  从类别 HO 随机选择建筑: Rat_health_Gaye
  从类别 LI 随机选择建筑: Eagle_public_Pearle
  从类别 OF 随机选择建筑: Eagle_office_Ryan
  从类别 UL 随机选择建筑: Hog_education_Rachael

🏢 建筑分组:
  训练建筑: 5 个 - ['Hog_lodging_Nikki', 'Rat_health_Gaye', 'Eagle_public_Pearle', 'Eagle_office_Ryan', 'Hog_education_Rachael']
  目标建筑: 1 个 - ['Gator_public_Leroy']

⚡ 加载电力数据...
  ✓ 完整电力数据: (17544, 25)
  ✓ 稀缺电力数据: (17544, 6)
  ✓ 稀缺训练集: (14035, 6) (前80%)
  ✓ 稀缺测试集: (3509, 6) (后20%)

🌤️  加载天气数据...
  训练建筑天气数据 (完整):
    ✓ Eagle: (17536, 5)
    ✓ Rat: (17539, 5)
    ✓ Hog: (17542, 5)
  目标建筑天气数据 (extreme):
    ✓ Gator: (17544, 5) (缺失率: 48.00%)

🔄 创建数据加载器...
  创建训练建筑数据集 (5 个建筑)...


2026-01-14 20:00:56,382 - INFO -  开始 DANN 适应...


  数据集: 5 个建筑, 17479 个有效样本
  创建目标建筑训练集数据集 (1 个建筑)...
  ⚠️  数据包含缺失 (缺失率: 60.00%)
  数据集: 1 个建筑, 13985 个有效样本
  创建目标建筑测试集数据集 (1 个建筑)...
  数据集: 1 个建筑, 3461 个有效样本

✅ 类别 CC 迁移学习数据加载完成!
训练样本: 31464
  - 训练建筑数据: 17479 样本
  - 目标建筑训练数据: 13985 样本
测试样本: 3461
批次大小: 32
缺失处理: forward_fill



2026-01-14 20:00:56,751 - INFO - 从目标域确定特征维度: 64
2026-01-14 20:00:56,751 - INFO - 开始域自适应迁移学习...
Epoch 1/10: 100%|███████████████████████████████████████████| 109/109 [00:44<00:00,  2.46it/s, loss=0.0009, tgt=0.0009]
2026-01-14 20:01:42,862 - INFO - ✅ 新最佳RMSE: 0.2161
2026-01-14 20:01:42,862 - INFO - Epoch 1/10 - Loss: 0.0104, RMSE: 0.2161
Epoch 2/10: 100%|███████████████████████████████████████████| 109/109 [00:44<00:00,  2.47it/s, loss=0.0012, tgt=0.0009]
2026-01-14 20:02:28,837 - INFO - ✅ 新最佳RMSE: 0.1846
2026-01-14 20:02:28,837 - INFO - Epoch 2/10 - Loss: 0.0152, RMSE: 0.1846
Epoch 3/10: 100%|███████████████████████████████████████████| 109/109 [00:43<00:00,  2.50it/s, loss=0.0016, tgt=0.0009]
2026-01-14 20:03:14,181 - INFO - Epoch 3/10 - Loss: 0.0184, RMSE: 0.1959
Epoch 4/10: 100%|███████████████████████████████████████████| 109/109 [00:43<00:00,  2.51it/s, loss=0.0023, tgt=0.0010]
2026-01-14 20:03:59,420 - INFO - ✅ 新最佳RMSE: 0.1627
2026-01-14 20:03:59,420 - INFO - Epoch 4/10 - Loss: 0


NoFeature_TL_CC_extreme 评估报告:
RMSD: 0.13507814705371857
MAPE: 37.18%
R²: 0.3456
CV-RMSE: 28.8312%
SD_real: 0.16698332130908966
SD_pred: 0.07781613618135452
CC: 0.6299

迁移学习评估:
A-distance: 0.9893094301223755
特征对齐: 0.8618223667144775
MMD: 0.08132171630859375

不确定性评估:
PICP: 37.34%
NMPIW: 0.16932006180286407
校准误差: 57.66%
UQS: 0.2467
平均不确定性: 0.03226441144943237
   Result: RMSD=0.1351, MAPE=37.18%

✅ 消融实验完成！结果已保存至 models/results_NoFeature_20260114_2010.csv


In [12]:
import csv
import json
import numpy as np
import torch
from datetime import datetime

# ==========================================
# 💾 单独保存扩展版 CSV 结果 (包含 SD_real, SD_pred)
# ==========================================

# 1. 定义辅助函数 (防止 tensor 或 numpy 类型导致写入报错)
def convert_to_basic_type(obj):
    if isinstance(obj, (np.ndarray, torch.Tensor)):
        if obj.size == 1: return float(obj.item())
        return str(obj.tolist())
    elif isinstance(obj, (np.float32, np.float64)): return float(obj)
    return str(obj) if obj is not None else "N/A"

# 2. 定义保存路径 (为了区别，加了后缀 _extended)
current_time = datetime.now().strftime('%Y%m%d_%H%M')
csv_path_extended = f"models/results_{EXPERIMENT_TAG}_{current_time}_extended.csv"

# 3. ✅ 更新表头，加入 SD_real 和 SD_pred
csv_header = [
    "Category", "Shortage", "Model", 
    "RMSD", "MAPE", "R2", "CV-RMSE", 
    "SD_real", "SD_pred",   # <--- 新增列在此
    "PICP", "NMPIW", "CE",
    "A-distance", "MMD"
]

csv_rows = []

# 4. 遍历内存中现有的 transfer_results 字典
if 'transfer_results' not in globals():
    print("❌ 错误: 内存中未找到 'transfer_results' 变量。请确保实验代码已运行。")
else:
    print(f"正在处理 {len(transfer_results)} 个类别的结果...")
    
    for cat, shortages in transfer_results.items():
        for shortage, res in shortages.items():
            # 提取 metrics 字典
            m = res.get('metrics', {})
            # 提取迁移指标 (如果 metrics 里没有，尝试从 transfer_metrics 取)
            t = res.get('transfer_metrics', {})
            if not t: t = m
                
            # 构建行数据
            row = [
                cat, 
                shortage, 
                f"Adaptive_{EXPERIMENT_TAG}",
                convert_to_basic_type(m.get('RMSD')), 
                convert_to_basic_type(m.get('MAPE')), 
                convert_to_basic_type(m.get('R2')), 
                convert_to_basic_type(m.get('CV-RMSE')),
                # ✅ 获取新增指标 SD_real 和 SD_pred
                convert_to_basic_type(m.get('SD_real')),
                convert_to_basic_type(m.get('SD_pred')),
                # 不确定性指标
                convert_to_basic_type(m.get('PICP')),
                convert_to_basic_type(m.get('NMPIW')),
                convert_to_basic_type(m.get('calibration_error')),
                # 迁移学习指标
                convert_to_basic_type(t.get('a_distance')),
                convert_to_basic_type(t.get('mmd'))
            ]
            csv_rows.append(row)

    # 5. 写入文件
    try:
        with open(csv_path_extended, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(csv_header)
            writer.writerows(csv_rows)
        print(f"\n✅ 扩展版结果已保存至: {csv_path_extended}")
        
        # 顺便打印前几行预览
        import pandas as pd
        if os.path.exists(csv_path_extended):
            df = pd.read_csv(csv_path_extended)
            print("\n数据预览 (前3行):")
            print(df[['Category', 'Shortage', 'RMSD', 'SD_real', 'SD_pred']].head(3))
            
    except Exception as e:
        print(f"❌ 保存 CSV 失败: {e}")

正在处理 6 个类别的结果...

✅ 扩展版结果已保存至: models/results_NoFeature_20260115_1108_extended.csv

数据预览 (前3行):
  Category Shortage      RMSD   SD_real   SD_pred
0       DO     mild  0.150209  0.145054  0.047489
1       DO    heavy  0.172970  0.145054  0.032917
2       DO  extreme  0.148280  0.145054  0.004533
